# Task Agents Playground

Ноутбук для ручной проверки `web_research_agent` и `document_research_agent`.

Сейчас основной сценарий здесь — задавать свои вопросы `web_research_agent` и смотреть нормализованный `AgentResult` со статусом, summary и источниками.

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

repo_root = Path.cwd()
if not (repo_root / ".env").exists() and (repo_root.parent / ".env").exists():
    repo_root = repo_root.parent

examples_root = repo_root / "examples"
if str(examples_root) not in sys.path:
    sys.path.insert(0, str(examples_root))

load_dotenv(repo_root / ".env")

print({
    "repo_root": str(repo_root),
    "examples_root": str(examples_root),
    "base_url": os.getenv("BASE_URL"),
    "chat_model": os.getenv("CHAT_MODEL"),
})

{'repo_root': 'd:\\Programming\\GitHub\\nirma', 'examples_root': 'd:\\Programming\\GitHub\\nirma\\examples', 'base_url': 'http://10.32.2.3:11434', 'chat_model': 'gpt-oss:120b'}


In [2]:
from src import (
    AgentTask,
    Store,
    create_document_research_agent,
    create_web_research_agent,
)


d:\Programming\GitHub\nirma\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
web_agent = create_web_research_agent()
document_agent_cache = {}


def display_result(title: str, result) -> None:
    display(Markdown(f"## {title}"))
    display(Markdown(f"**status:** `{result.status}`"))
    if result.summary:
        display(Markdown(result.summary))

    if result.sources:
        lines = ["**Источники:**"]
        for source in result.sources:
            label = source.title or source.locator
            locator = source.locator or ""
            snippet = source.snippet or ""
            lines.append(f"- [{label}]({locator})")
            if snippet:
                lines.append(f"  - {snippet}")
        display(Markdown("\n".join(lines)))

    print(result.model_dump_json(indent=2, ensure_ascii=False))


def ask_web(question: str, context: dict | None = None, constraints: dict | None = None):
    task = AgentTask(
        kind="web_research",
        query=question,
        context=context or {},
        constraints=constraints or {},
    )
    return web_agent.execute(task)


def get_document_agent(document_path: str):
    if not document_path:
        raise ValueError("Укажи путь к PDF или DOCX файлу.")
    if document_path not in document_agent_cache:
        store = Store(document_path)
        document_agent_cache[document_path] = create_document_research_agent(store=store)
    return document_agent_cache[document_path]


def ask_document(question: str, document_path: str, context: dict | None = None, constraints: dict | None = None):
    agent = get_document_agent(document_path)
    task = AgentTask(
        kind="document_research",
        query=question,
        context=context or {},
        constraints=constraints or {},
        metadata={"document_path": document_path},
    )
    return agent.execute(task)


## Свой вопрос к web-агенту

Меняй текст в `web_question` и просто перезапускай ячейку.

Первый запрос может выполняться заметно дольше обычной ячейки, потому что агент собирает внешние источники.

In [8]:
web_question = (
    "Как можно развивать Гатчину с учетом ее роли в агломерации Санкт-Петербурга? "
    "Кратко назови 5 стратегических направлений и отдельно отметь транспорт, "
    "экономику и городскую среду."
)

web_result = ask_web(web_question)
display_result("Результат web-агента", web_result)

## Результат web-агента

**status:** `success`

Пять стратегических направлений развития — улучшение транспортных связей (расширение автодороги E95 и интеграция с кольцевой A120), развитие туристической инфраструктуры вокруг Большого Гатчинского дворца, создание бизнес‑парков и поддержка малого и среднего предпринимательства как административного центра, расширение жилой застройки для притока населения из Санкт‑Петербурга, а также формирование и озеленение городской среды. Транспорт — проектировать новые железнодорожные и автобусные сообщения, используя расположение Гатчины на E95 и пересечение с A120 (см. Wikipedia‑фрагменты). Экономика — использовать статус административного центра и культурное наследие для привлечения инвестиций в туризм и бизнес‑парки (источники о городе и дворце). Городская среда — внедрять программы озеленения, благоустройства исторических кварталов и создания новых общественных пространств. (Ответ предварите...

**Источники:**
- [Gatchina](https://en.wikipedia.org/wiki/Gatchina)
  - Gatchina (Russian: Га́тчина, IPA: [ˈɡatːɕɪnə]) is a town and the administrative center of Gatchinsky District in Leningrad Oblast, Russia. It lies 45 kilometers (28 mi) south-southwest of St. Petersburg, along the E95 highway which links Saint Petersburg and Pskov. Population: 92,937 (2010 census); 88,420 (2002 cens...
- [Great Gatchina Palace](https://en.wikipedia.org/wiki/Great_Gatchina_Palace)
  - The Great Gatchina Palace (Russian: Большой Гатчинский дворец) is a palace in Gatchina, Leningrad Oblast, Russia. It was built from 1766 to 1781 by Antonio Rinaldi for Count Grigori Grigoryevich Orlov, who was a favourite of Catherine the Great, in Gatchina, a suburb of the royal capital Saint Petersburg. The Gatchi...
- [Gatchinsky District - Wikipedia](https://en.wikipedia.org/wiki/Gatchinsky_District)
  - Another one originates inSaintPetersburg, passesGatchina, and proceeds to Luga and Pskov . ...Gatchina, it crosses the A120 highway, which ...
- [Transportation | St.Petersburg Travel Guide](https://guidetopetersburg.com/posts/transportation/)
  - You can get aroundSaintPetersburgby publictransport(metro, buses, trolleybuses, and trams) and by private commercialtransport(buses ...
- [Gatchina | St.Petersburg Travel Guide](https://guidetopetersburg.com/posts/gatchina/)
  - Petersburg, doesn ’ t enjoy the status of a tourism hot spot, but is still anyway a lovely place to make a day trip.

{
  "task_id": "9e94f836-cef3-4a5a-b57f-5b254ba05f98",
  "agent_name": "web_research_agent",
  "status": "success",
  "summary": "Пять стратегических направлений развития — улучшение транспортных связей (расширение автодороги E95 и интеграция с кольцевой A120), развитие туристической инфраструктуры вокруг Большого Гатчинского дворца, создание бизнес‑парков и поддержка малого и среднего предпринимательства как административного центра, расширение жилой застройки для притока населения из Санкт‑Петербурга, а также формирование и озеленение городской среды. Транспорт — проектировать новые железнодорожные и автобусные сообщения, используя расположение Гатчины на E95 и пересечение с A120 (см. Wikipedia‑фрагменты). Экономика — использовать статус административного центра и культурное наследие для привлечения инвестиций в туризм и бизнес‑парки (источники о городе и дворце). Городская среда — внедрять программы озеленения, благоустройства исторических кварталов и создания новых общественных про

## Быстрый шаблон для своих вопросов

Если хочешь просто подставлять новые формулировки, используй эту ячейку.

In [9]:
# Примеры:
# custom_question = "Какие сильные и слабые стороны есть у Гатчины как города-спутника Петербурга?"
# custom_question = "Какие городские функции логично усиливать в Гатчине в ближайшие 10 лет?"

custom_question = "Какие городские функции логично усиливать в Гатчине в ближайшие 10 лет?"

custom_result = ask_web(custom_question)
display_result("Результат по пользовательскому вопросу", custom_result)

## Результат по пользовательскому вопросу

**status:** `success`

В ближайшие 10 лет в Гатчине целесообразно усилить функции, связанные с транспортной инфраструктурой и туризмом. Город расположен на трассе E95, соединяющей Санкт‑Петербург и Псков, а также имеет важный железнодорожный узел Varshavskaya Station, что делает развитие автодорожного и железнодорожного сообщения приоритетом. Кроме того, наличие Великого Гатчинского дворца привлекает туристов, поэтому стоит расширять туристическую и культурно‑развлекательную инфраструктуру. Укрепление этих функций будет поддерживать роль Гатчины как административного центра района и её связь с мегаполисом‑соседом. (Ответ предварительный, поскольку источники ограничены.)

**Источники:**
- [Gatchina](https://en.wikipedia.org/wiki/Gatchina)
  - Gatchina (Russian: Га́тчина, IPA: [ˈɡatːɕɪnə]) is a town and the administrative center of Gatchinsky District in Leningrad Oblast, Russia. It lies 45 kilometers (28 mi) south-southwest of St. Petersburg, along the E95 highway which links Saint Petersburg and Pskov. Population: 92,937 (2010 census); 88,420 (2002 cens...
- [Great Gatchina Palace](https://en.wikipedia.org/wiki/Great_Gatchina_Palace)
  - The Great Gatchina Palace (Russian: Большой Гатчинский дворец) is a palace in Gatchina, Leningrad Oblast, Russia. It was built from 1766 to 1781 by Antonio Rinaldi for Count Grigori Grigoryevich Orlov, who was a favourite of Catherine the Great, in Gatchina, a suburb of the royal capital Saint Petersburg. The Gatchi...
- [Real-time publictransportinGatchinatracking on Yandex Maps](https://yandex.com/maps/10867/gatchina/transport/)
  - Routes, stops, and real-time tracking: keep tabs on your bus, minibus, trolleybus, or tram inGatchina. PublictransportinGatchinaon Yandex Maps.
- [GatchinaVarshavskaya Station (2025) – Best of TikTok, Instagram...](https://airial.travel/attractions/russia/gatchina-varshavskaya-station-kZhqpoQS)
  - Understanding theTransportNetwork.GatchinaVarshavskaya Station serves as a critical nexus for both rail and bustransportationin theGatchinadistrict.
- [Gatchina - Wikipedia](https://en.wikipedia.org/wiki/Gatchina)
  - December 4, 2025 -Gatchina (Russian: Га́тчина, IPA: [ˈɡatːɕɪnə]) is a town and the administrative center of Gatchinsky District in Leningrad Oblast, Russia. It lies 45 kilometers (28 mi) south-southwest of St. Petersburg, along the E95 highway which links Saint Petersburg and Pskov.

{
  "task_id": "4c0ef8bf-f86b-43fd-9121-f8a79861a7c0",
  "agent_name": "web_research_agent",
  "status": "success",
  "summary": "В ближайшие 10 лет в Гатчине целесообразно усилить функции, связанные с транспортной инфраструктурой и туризмом. Город расположен на трассе E95, соединяющей Санкт‑Петербург и Псков, а также имеет важный железнодорожный узел Varshavskaya Station, что делает развитие автодорожного и железнодорожного сообщения приоритетом. Кроме того, наличие Великого Гатчинского дворца привлекает туристов, поэтому стоит расширять туристическую и культурно‑развлекательную инфраструктуру. Укрепление этих функций будет поддерживать роль Гатчины как административного центра района и её связь с мегаполисом‑соседом. (Ответ предварительный, поскольку источники ограничены.)",
  "sources": [
    {
      "type": "wikipedia",
      "title": "Gatchina",
      "locator": "https://en.wikipedia.org/wiki/Gatchina",
      "snippet": "Gatchina (Russian: Га́тчина, IPA: [ˈɡatːɕɪnə]) is a town and

## Настройка документа

Если захочешь отдельно проверить `document_research_agent`, укажи путь к своему PDF или DOCX.

In [6]:
DOCUMENT_PATH = os.getenv("NIRMA_DOCUMENT_PATH", "")
DOCUMENT_PATH

''

## Свой вопрос к document-агенту

Сначала заполни `DOCUMENT_PATH`, затем меняй `document_question` и запускай ячейку.

In [7]:
document_question = "Какие ключевые стратегические приоритеты и направления развития упоминаются в этом документе?"

if not DOCUMENT_PATH:
    raise ValueError("Сначала заполни DOCUMENT_PATH или задай переменную окружения NIRMA_DOCUMENT_PATH.")

document_result = ask_document(document_question, DOCUMENT_PATH)
display_result("Результат document-агента", document_result)

ValueError: Сначала заполни DOCUMENT_PATH или задай переменную окружения NIRMA_DOCUMENT_PATH.